# Solutions for Week 7 activities: reshaping and merging data

# Setup

In [10]:
# packages
import pandas as pd
import numpy as np
import os
import re

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"



Load the data `childnoinsur_complete.csv` data

In [8]:
os.getcwd()

'/Users/rj545/Dropbox/PPOL5203_fall2026/activities'

In [5]:
df = pd.read_csv("../datasets/childnoinsur_complete.csv")
df.head()


,State,Race,Year,uninsured_rate,Census_Division
0,Wyoming,Non-Hispanic White,2018,0.07,Mountain
1,Wyoming,Hispanic or Latino,2018,0.06,Mountain
2,Wyoming,Black or African American,2018,0.02,Mountain
3,Wyoming,Asian and Pacific Islander,2018,0.02,Mountain
4,Wisconsin,Two or more races,2018,0.02,East North Central


# Activity 1: reshaping data

The data as loaded is already long-format data, which repeats each state across different race/ethnicity groups and years of uninsurance

A. Subset to the year 2024 and pivot the data to wide. This should have one row for each state and one column for the uninsurance rate in each racial/ethnic group. Store this data as `df_2024_wide`

B. Use list comprehension to remove spaces/punctuation from the column names. You may want to use `sub` in the re module: https://www.geeksforgeeks.org/python/re-sub-python-regex/

C. Using `df_2024_wide`, create a new column `bw_ratio` that divides the Black uninsurance rate by the white uninsurance rate 

D. Sort states from highest (reflecting much higher uninsurance rates among Black children than white children) to lowest

In [16]:
# A. subset and reshape
df_2024_wide = pd.pivot(df[df.Year == 2024],
                       index = 'State',
                       columns = 'Race',
                       values = 'uninsured_rate').reset_index()
df_2024_wide.head()

## B. clean up column names
df_2024_wide.columns = [re.sub("\\s|-", "_", x.lower()) for x in df_2024_wide.columns]

## C. find the ratio 
df_2024_wide['bw_ratio'] = df_2024_wide.black_or_african_american / \
                            df_2024_wide.non_hispanic_white
    
## D. print
df_2024_wide[['state',
              'black_or_african_american',
              'non_hispanic_white', 
              'bw_ratio']].sort_values(by = 'bw_ratio', ascending = False)

Race,State,American Indian or Alaska Native,Asian and Pacific Islander,Black or African American,Hispanic or Latino,Non-Hispanic White,Two or more races
0,Alabama,0.05,0.03,0.04,0.08,0.03,0.04
1,Alaska,0.10,0.11,NaN,0.07,0.07,0.10
2,Arizona,0.22,0.05,0.13,0.11,0.06,0.10
3,Arkansas,NaN,NaN,0.07,0.16,0.06,0.09
4,California,0.04,0.02,0.03,0.04,0.02,0.03


,state,black_or_african_american,non_hispanic_white,bw_ratio
22,Minnesota,0.06,0.02,3.000000
20,Massachusetts,0.06,0.02,3.000000
15,Kansas,0.10,0.04,2.500000
29,New Jersey,0.05,0.02,2.500000
2,Arizona,0.13,0.06,2.166667
19,Maryland,0.04,0.02,2.000000
4,California,0.03,0.02,1.500000
5,Colorado,0.06,0.04,1.500000
6,Connecticut,0.03,0.02,1.500000
41,Tennessee,0.06,0.04,1.500000


# Activity 2

Load the following dataset: `child_poverty.csv`

In [17]:
cp = pd.read_csv("../datasets/child_poverty.csv")

In [18]:
cp.head()

,state,year,race_ethnicity,child_poverty_rate
0,Alabama,2023,American Indian or Alaska Native,NaN
1,Alabama,2023,Asian and Pacific Islander,0.08
2,Alabama,2023,Black or African American,0.41
3,Alabama,2023,Hispanic or Latino,0.27
4,Alabama,2023,Non-Hispanic White,0.11


A. Use .head() and value_counts() to explore the column names and years represented in the data 



In [19]:
cp.year.value_counts()

2023    350
2024    350
Name: year, dtype: int64

B. Subset the original `childnoinsur_complete.csv` data (`ci` for shorthand) to 2023 and 2024 and and
synchronize the following column names between the two datasets:
    
- State / state
- Year / year 
- Race / race_ethnicity

In [29]:
ci = df[df.Year.isin([2023, 2024])].copy()
ci.columns = ["race_ethnicity" if col == "Race"
       else col.lower() for col in ci.columns]


B. Conduct a merge where the:
    
- Left hand data is that subsetted / column renamed child uninsurance data
- Right hand data is the child poverty data (`cp` for shorthand)

Merge by state, year, and race/ethnicity group keeping all rows in the left hand-side data (left join)

Store the result as `cileft` and print the head and dimensions. 

How many rows were lost from the `cp` child poverty data? Why? (hint: it relates to the child uninsurance dataset's exclusions of missing values)

In [31]:

## merge
cileft = pd.merge(ci, cp, 
                 on = ['state', 'year', 'race_ethnicity'],
                 how = 'left')
cileft.head()
cileft.shape
cp.shape

,state,race_ethnicity,year,uninsured_rate,census_division,child_poverty_rate
0,Alaska,American Indian or Alaska Native,2024,0.10,Pacific,0.31
1,Alaska,Asian and Pacific Islander,2024,0.11,Pacific,NaN
2,Alaska,Hispanic or Latino,2024,0.07,Pacific,NaN
3,Alaska,Non-Hispanic White,2024,0.07,Pacific,0.05
4,Alaska,Two or more races,2024,0.10,Pacific,0.14


(483, 6)

(700, 4)

C. Conduct the same merge as part B but treat the `cp` child poverty data as the left hand side of the data, preserving all rows. 

Store the result as `cpleft`

Print the head and dimensions of the resulting data. Why were no rows dropped?

In [32]:
cpleft = pd.merge(cp, ci, 
                 on = ['state', 'year', 'race_ethnicity'],
                 how = 'left')
cpleft.head()
cpleft.shape
cp.shape

,state,year,race_ethnicity,child_poverty_rate,uninsured_rate,census_division
0,Alabama,2023,American Indian or Alaska Native,NaN,0.04,East South Central
1,Alabama,2023,Asian and Pacific Islander,0.08,NaN,NaN
2,Alabama,2023,Black or African American,0.41,0.05,East South Central
3,Alabama,2023,Hispanic or Latino,0.27,0.10,East South Central
4,Alabama,2023,Non-Hispanic White,0.11,0.02,East South Central


(700, 6)

(700, 4)

D. You notice that the previous merge lead to missing values for census division for rows where the child uninsurance rate is missing. Since that is likely non-missing across some rows within a state, do the following:

- Use grouping and aggregation to find the census division of each state, storing the result as `state_divisions` (these divisions are constant across all years and race/ethnicity groups)
- Drop the `census_division` column from the `cpleft` dataset
- Merge `state_divisions` and `cpleft` using the `state` column to add a more complete version of the Census Division variable to that dataset. Store the result as `cpleft_wdiv`

In [35]:
state_divisions = ci.groupby('state')['census_division'].agg(lambda x: x.mode().iloc[0]).reset_index()

cpleft = cpleft.drop(columns = 'census_division')

cpleft_wdiv = pd.merge(cpleft, state_divisions, on = 'state', how = 'left')
cpleft_wdiv.head()

,state,year,race_ethnicity,child_poverty_rate,uninsured_rate,census_division
0,Alabama,2023,American Indian or Alaska Native,NaN,0.04,East South Central
1,Alabama,2023,Asian and Pacific Islander,0.08,NaN,East South Central
2,Alabama,2023,Black or African American,0.41,0.05,East South Central
3,Alabama,2023,Hispanic or Latino,0.27,0.10,East South Central
4,Alabama,2023,Non-Hispanic White,0.11,0.02,East South Central


E. Using `cpleft_wdiv` find the mean child poverty rate and child uninsured rate by census division (across all states and race/ethnicity groups). What are some things you notice?

**Note**: the default behavior of mean is to take the mean ignoring missing values

In [39]:
cpleft_wdiv.groupby('census_division')[['child_poverty_rate', 
                                       'uninsured_rate']].mean().reset_index().sort_values(by = 
                                        'child_poverty_rate',
                                        ascending = False)

,census_division,child_poverty_rate,uninsured_rate
1,East South Central,0.224762,0.065385
8,West South Central,0.219149,0.094048
0,East North Central,0.191000,0.047857
2,Middle Atlantic,0.172778,0.041765
6,South Atlantic,0.154699,0.062105
3,Mountain,0.145000,0.074667
7,West North Central,0.143276,0.055000
5,Pacific,0.140741,0.043137
4,New England,0.123444,0.027609
